# Grip and the speed limit

**Why does the car spin at Parabolica?**

Unit 5's agent drives a clean 20 m/s on racekit's kinematic car, on every track.
That car is a model with one big lie in it: its tyres never slide.
Whatever the wheel angle, the car follows the arc it asks for, at any speed.

Unit 6 races a car whose tyres **can** slide: racekit's `DynamicCar`, a dynamic bicycle with tyres that saturate.
Here is Unit 5's agent, unchanged, on that car, with grip μ = 1 (dry tarmac), on `test-circuit` and on `monza-like`.
(`agents/unit-06.py` with `SPEED_MODE = "flat"` races exactly like `agents/unit-05.py`; Unit 6's milestone checks it to the tick.)
Both races run to the 1500 s cap, so this cell takes about half a minute.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import DynamicCar, KinematicCar, carla_like
from racekit.runner import run_race
from racekit.tracks import load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_06 = AGENTS / "unit-06.py"
workdir = Path(mkdtemp(prefix="roar-unit-06-"))
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
OFFSET_TOL = 0.01  # m, the book's tolerance on a max |cross-track error|
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on max |cross-track error| (section 6)


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def spins(result):
    # How many of the race's respawns were spin-outs (the others hit the wall).
    return sum(cause == "spin" for cause in result.respawn_causes)


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def summary(result):
    if result.finished and result.collisions == 0:
        return f"finished, {result.total_time:6.2f} s, 0 collisions, max |offset| {max_offset(result):.2f} m"
    finished = "finished" if result.finished else "did not finish"
    return f"{result.collisions} collisions ({spins(result)} spin-outs), {finished} at {result.total_time:.2f} s"


UNIT_05_AGENT = variant(UNIT_06, "unit-06-flat", SPEED_MODE="flat")  # Unit 5's agent, to the tick
hook = {}
for track in ("test-circuit", "monza-like"):
    hook[track] = run_race(UNIT_05_AGENT, track, car=DynamicCar(), seed=0, max_seconds=1500.0)
    print(f"Unit 5's agent on DynamicCar(), {track:>12}: {summary(hook[track])}")
spin_race, wide_race = hook["test-circuit"], hook["monza-like"]
assert (spins(spin_race), spin_race.collisions, spin_race.finished) == (40, 40, False)
assert (spins(wide_race), wide_race.collisions, wide_race.finished) == (0, 23, False)

On `test-circuit` the car **spins** 40 times and never finishes; on `monza-like` it never spins but **runs wide** into the wall 23 times, mostly at the long fast bends like Monza's Parabolica.
On the `oval` it is still clean, 216.50 s (probe record (plan roar-006)): the oval's bends are gentle enough.

Where does it happen?
Each respawn is marked at the last position the car had before it.

In [ ]:
def respawn_points(result):
    # The (x, y) of the last telemetry row before each respawn.
    telemetry = result.telemetry
    rows = [int(np.nonzero(telemetry["t"] < time)[0][-1]) for time in result.respawn_times]
    return telemetry["x"][rows], telemetry["y"][rows]


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (track, result) in zip(axes, hook.items()):
    centreline = load_track(track)
    ax.plot(centreline.x, centreline.y, "k-", linewidth=0.8)
    px, py = respawn_points(result)
    ax.scatter(px, py, color="tab:red", s=18, label="spin-out" if spins(result) else "hit the wall")
    ax.set_aspect("equal")
    ax.set_title(f"{track}: Unit 5's agent at 20 m/s on DynamicCar()")
    ax.legend()
plt.show()

The marks sit in the **tight corners** of `test-circuit` and the **long bends** of `monza-like`: two symptoms, one cause.
**The car asks the tyres for more sideways force than they can give.**
On the kinematic car that never happened, because its tyres had no limit.

This unit builds the model that explains it, measures the limit, and turns it into a speed limit the agent can respect:

1. the **dynamic bicycle**: where the sideways force comes from (slip angles);
2. the **friction circle**: why that force has a ceiling, and what happens at the ceiling;
3. the **cornering speed limit** v_max = √(a_lat / |κ|), built from Unit 2's curvature ahead;
4. how close to the ceiling the agent can drive, on the book's model of the competition car.

## 1. The dynamic bicycle

Unit 3's kinematic bicycle assumes each wheel rolls exactly where it points.
A real tyre makes a sideways force only by **slipping** a little: the wheel points one way and the contact patch moves in a slightly different direction.
The angle between the two is the **slip angle**.

Put the car's velocity in its own frame: v_x forward, v_y to the left, and a yaw rate r.
The front axle is l_f ahead of the centre of gravity and the rear axle l_r behind it (l_f + l_r = L), so the velocity at each axle is (v_x, v_y + l_f r) and (v_x, v_y − l_r r), and

$$\alpha_f = \operatorname{atan2}(v_y + l_f r,\; v_x) - \delta, \qquad \alpha_r = \operatorname{atan2}(v_y - l_r r,\; v_x).$$

Each axle's lateral force is a **saturating** function of its slip angle:

$$F_y = -\mu F_z \,\sin\!\big(C \arctan(B\alpha)\big),$$

with F_z the axle's load and B, C two shape constants (racekit's `DynamicCar` uses B = 10, C = 1.6).
Small α gives a force proportional to α; large α gives at most μF_z, and past the peak the force **falls**.
`DynamicCar` splits the weight between the axles statically: F_zf = m g l_r / L and F_zr = m g l_f / L.

Newton's laws in the car frame give the dynamic bicycle (δ the front wheel angle, F_xf and F_xr the axles' drive or brake forces):

$$m(\dot v_x - v_y r) = F_{xf}\cos\delta - F_{yf}\sin\delta + F_{xr},$$
$$m(\dot v_y + v_x r) = F_{xf}\sin\delta + F_{yf}\cos\delta + F_{yr},$$
$$I_z\,\dot r = l_f\,(F_{xf}\sin\delta + F_{yf}\cos\delta) - l_r F_{yr}.$$

The last line is the **yaw equation**: the front force turns the nose in, the rear force turns it back, each with its lever arm.
The state has grown from Unit 3's (x, y, ψ, v) to (x, y, ψ, v_x, v_y, r), plus the actual wheel angle (a steering motor turns it at most 360°/s) and a one-tick actuator lag.

Here is the tyre curve, with its peak:

In [ ]:
car = DynamicCar()  # mu = 1, the book's dry-tarmac model
B, C = car.tyre_b, car.tyre_c
print(f"DynamicCar(): m = {car.mass:.0f} kg, I_z = {car.yaw_inertia:.0f} kg m^2, l_f = {car.lf:.3f} m, "
      f"l_r = {car.lr:.3f} m, mu = {car.mu}, lag {car.lag_ticks} tick")
print(f"static loads: front {car.fz_f:.0f} N, rear {car.fz_r:.0f} N")


def tyre_fraction(alpha):
    # F_y / (mu F_z) = -sin(C atan(B alpha)); atan(u) is written atan2(u, 1)
    return -math.sin(C * math.atan2(B * alpha, 1.0))


alpha_peak = math.tan(math.pi / (2.0 * C)) / B  # C atan(B alpha) = pi/2 at the peak
alphas = np.radians(np.linspace(-25.0, 25.0, 501))
fractions = np.array([tyre_fraction(alpha) for alpha in alphas])
print(f"the peak: |alpha| = {alpha_peak:.4f} rad = {math.degrees(alpha_peak):.2f} deg; "
      f"past it, at 25 deg, the force has fallen to {abs(tyre_fraction(math.radians(25.0))):.3f} of mu F_z")
assert abs(abs(tyre_fraction(alpha_peak)) - 1.0) < 1e-12
assert abs(abs(alphas[np.argmax(-fractions)]) - alpha_peak) < math.radians(0.1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.degrees(alphas), fractions)
ax.axvline(math.degrees(alpha_peak), color="grey", linestyle=":")
ax.axvline(-math.degrees(alpha_peak), color="grey", linestyle=":")
ax.set_xlabel("slip angle alpha (deg)")
ax.set_ylabel("F_y / (mu F_z)")
ax.set_title("DynamicCar's tyre: linear for small slip, saturating at mu F_z")
plt.show()

### At low speed it is Unit 3's car

In a gentle turn the forces needed are small, so the slip angles are tiny, each axle's velocity points almost exactly along its wheel, and that **is** the kinematic bicycle's assumption.
So at low speed the two models should agree.
Race them side by side: both start at 5 m/s, hold a small right steer (−0.02, a wheel angle of 1.4°) for 10 s, with the throttle that balances the drag.
`KinematicCar(lag_ticks=1)` matches the dynamic car's one-tick lag.
Compare the yaw rate and the **path curvature** κ = r / v.
(The two cars report different points, the kinematic car its rear axle and the dynamic car its centre of gravity, so their positions are not compared.)

In [ ]:
DT = 0.05  # s: one control tick
C_DRAG, A_THROTTLE = 4.0e-4, 6.0  # Unit 3's longitudinal map (DynamicCar uses the same values)
STEER = -0.02


def twin_drive(speed, seconds=10.0):
    # Drive both cars at `speed` with STEER held; return (kinematic r, dynamic r, dynamic v, dynamic sideslip).
    dynamic, kinematic = DynamicCar(), KinematicCar(lag_ticks=1)
    dynamic.reset(0.0, 0.0, 0.0, speed)
    kinematic.reset(0.0, 0.0, 0.0, speed)
    hold = C_DRAG * speed * speed / A_THROTTLE
    for _ in range(int(round(seconds / DT))):
        dynamic.advance(hold, STEER, 0.0, 0.0, 0.0, DT, 10)
        kinematic.advance(hold, STEER, 0.0, 0.0, 0.0, DT, 10)
    kinematic_r = kinematic.v * math.tan(-STEER * kinematic.max_steer) / kinematic.wheelbase  # Unit 3: v tan(delta) / L
    return kinematic_r, dynamic.r, dynamic.v, dynamic.sideslip()


print(f"{'v (m/s)':>8}{'r kinematic':>13}{'r dynamic':>11}{'kappa kin':>11}{'kappa dyn':>11}{'v dyn after 10 s':>18}")
twins = {}
for speed in (5.0, 10.0, 20.0):
    kinematic_r, dynamic_r, dynamic_v, beta = twin_drive(speed)
    twins[speed] = (kinematic_r, dynamic_r, dynamic_v)
    print(f"{speed:8.0f}{kinematic_r:13.5f}{dynamic_r:11.5f}{kinematic_r / speed:11.6f}{dynamic_r / dynamic_v:11.6f}{dynamic_v:18.2f}")
kinematic_r, dynamic_r, _ = twins[5.0]
assert abs(dynamic_r / kinematic_r - 1.0) < 0.002  # at 5 m/s the yaw rates agree to 0.2%
for speed, (kinematic_r, dynamic_r, dynamic_v) in twins.items():
    assert abs((dynamic_r / dynamic_v) / (kinematic_r / speed) - 1.0) < 0.002  # the same path curvature
assert twins[20.0][2] < 19.5  # but at 20 m/s the dynamic car has lost speed

At 5 m/s the two cars turn at the same rate, to a fraction of a percent: **the dynamic bicycle reduces to the kinematic bicycle at low speed.**

Two things in the table are worth a second look.

- The path curvature agrees at every speed in the table.
  That is a property of this model, not of cars: for small slip the axle's force is −C_α α with **cornering stiffness** C_α = B·C·μF_z, so C_α is proportional to the axle's load.
  The textbook understeer gradient is (m / L)(l_r / C_αf − l_f / C_αr), and with F_zf ∝ l_r and F_zr ∝ l_f both terms equal L / (B C μ m g): the gradient is **zero**, a neutral-steer car.
  A real car is usually built to understeer a little.
- At 20 m/s the dynamic car **slows down** with the throttle that held the kinematic car's speed: the steered front tyre's force has a component against the motion (the −F_yf sin δ term), and the throttle no longer balances it.
  Unit 4's speed PI would simply add throttle; here the throttle was fixed on purpose.

So far the two cars disagree only in small ways.
The big difference is what happens when the corner asks for more force than the tyres have.

## 2. The friction circle

A tyre's grip is one budget for both directions.
The total horizontal force an axle can carry is μF_z, in any direction:

$$F_x^2 + F_y^2 \le (\mu F_z)^2 .$$

`DynamicCar` enforces it per axle: the drive or brake force is clipped to ±μF_z first, and the lateral force is scaled to what is left, √((μF_z)² − F_x²).
Add the two axles and divide by the mass: the car's acceleration in the ground plane can never leave a circle of radius μg,

$$a_{\text{lat}}^2 + a_{\text{long}}^2 \le (\mu g)^2, \qquad \mu g = 9.81\ \text{m/s}^2 \text{ at } \mu = 1 .$$

That is the **friction circle**.
In a steady turn at speed v on a path of curvature κ the lateral acceleration is v²κ, so the circle is a speed limit.

### A skidpad

Test it the way engineers do, on a **skidpad**: hold the steering fixed and creep the speed up until the car can no longer hold the circle.
The steer is −0.1 (a wheel angle of 7°), the car starts at 5 m/s, and the speed goal rises by 0.2 m/s every second; a P loop plus the drag-balancing throttle follows it.
Every tick we compute the car-frame accelerations from its velocities, exactly as in the equations above:
a_long = v̇_x − v_y r and a_lat = v̇_y + v_x r (the tick's change over DT).
The run stops at the first tick on which `spun_out()` says the car has spun: its **sideslip** β = atan2(v_y, |v_x|) is beyond 25° at more than 5 m/s.

In [ ]:
G = 9.81  # m/s^2


def skidpad(steer=-0.1, ramp=0.2, v0=5.0, mu=1.0, max_ticks=4000):
    # Fixed steer, a slow speed ramp; per tick: (t, v, a_long, a_lat, beta, alpha_f, alpha_r). Stops at a spin-out.
    car = DynamicCar(mu=mu)
    car.reset(0.0, 0.0, 0.0, v0)
    rows = np.zeros((max_ticks, 7))
    for n in range(max_ticks):
        goal = v0 + ramp * n * DT
        throttle = min(max(0.5 * (goal - car.v) + C_DRAG * car.v * car.v / A_THROTTLE, 0.0), 1.0)
        vx, vy = car.vx, car.vy
        car.advance(throttle, steer, 0.0, 0.0, 0.0, DT, 10)
        a_long = (car.vx - vx) / DT - car.vy * car.r
        a_lat = (car.vy - vy) / DT + car.vx * car.r
        alpha_f = math.atan2(car.vy + car.lf * car.r, car.vx) - car.delta
        alpha_r = math.atan2(car.vy - car.lr * car.r, car.vx)
        rows[n] = (n * DT + DT, car.v, a_long, a_lat, car.sideslip(), alpha_f, alpha_r)
        if car.spun_out():
            return rows[: n + 1], True, car.mu_now
    return rows, False, car.mu_now


pad, spun, mu = skidpad()
t, v, a_long, a_lat, beta = pad[:, 0], pad[:, 1], pad[:, 2], pad[:, 3], pad[:, 4]
peak = int(np.argmax(np.abs(a_lat)))
total = np.hypot(a_long, a_lat)
print(f"mu = {mu}, mu g = {mu * G:.2f} m/s^2")
print(f"peak |a_lat| = {abs(a_lat[peak]):.3f} m/s^2 = {abs(a_lat[peak]) / (mu * G):.3f} mu g, at v = {v[peak]:.2f} m/s, t = {t[peak]:.1f} s")
print(f"there: a_long = {a_long[peak]:+.3f} m/s^2, |a| = {total[peak]:.3f} m/s^2 = {total[peak] / (mu * G):.4f} mu g")
print(f"slip angles there: front {math.degrees(pad[peak, 5]):+.2f} deg, rear {math.degrees(pad[peak, 6]):+.2f} deg "
      f"(the tyre's peak is at {math.degrees(alpha_peak):.2f} deg)")
print(f"spun out: {spun}, at t = {t[-1]:.2f} s and v = {v[-1]:.2f} m/s, sideslip {math.degrees(beta[-1]):+.1f} deg")
assert spun and mu == 1.0
assert 0.97 * mu * G < abs(a_lat[peak]) < mu * G
assert 0.99 * mu * G < total[peak] <= 1.001 * mu * G
assert abs(pad[peak, 6]) > alpha_peak > abs(pad[peak, 5]) > 0.9 * alpha_peak  # the rear is past the peak

The lateral acceleration climbs with v² until it **flattens just under μg**, and a few seconds later the car spins.
At the peak the total acceleration is μg to a fraction of a percent: the lateral part is a little under μg because a little of the circle goes to the throttle that keeps the speed rising.
The slip angles at the peak are at the tyre curve's peak, and the rear's is just past it: there is no more force to be had by slipping more, and past the peak a little more slip gives *less* force, so the rear lets go and the nose swings in.

The plot shows both views: a_lat against the speed, with the v²κ line it follows until it cannot, and the (a_long, a_lat) pairs inside the circle of radius μg.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
early = v < 10.0
kappa_pad = float(np.median(np.abs(a_lat[early]) / v[early] ** 2))  # the skidpad circle's curvature at low speed
left.plot(v, np.abs(a_lat), label="|a_lat| measured")
left.plot(v, kappa_pad * v ** 2, ":", label=f"v^2 kappa, kappa = {kappa_pad:.4f} 1/m")
left.axhline(mu * G, color="grey", linestyle="--", label="mu g")
left.set_xlabel("v (m/s)")
left.set_ylabel("|a_lat| (m/s^2)")
left.set_title("skidpad: steer -0.1, speed ramp 0.2 m/s per s")
left.legend()
angle = np.linspace(0.0, 2.0 * math.pi, 200)
right.plot(mu * G * np.cos(angle), mu * G * np.sin(angle), "k-", linewidth=0.8, label="the circle, radius mu g")
right.plot(a_long, a_lat, ".", markersize=2, label="(a_long, a_lat) every tick")
right.set_aspect("equal")
right.set_xlabel("a_long (m/s^2)")
right.set_ylabel("a_lat (m/s^2)")
right.set_title("the friction circle")
right.legend()
plt.show()

Until the limit, |a_lat| follows v²κ with the skidpad circle's low-speed curvature: this neutral-steer car holds its circle.
Then it flattens against μg while the speed goal keeps rising, the rear slip angle passes the tyre's peak, and the last few ticks are the spin itself, when the car is no longer in a steady turn at all.

This model has limits, and they are labelled: the axle loads are static (no **load transfer** forward under braking or outward in a corner), there are no gears and no tyre temperature.
A real car under braking puts more load on the front tyres and less on the rear, which is why a real car can spin when it brakes in a corner.
Challenge (a) in the exercises asks what this model does instead.

## 3. The cornering speed limit

If the grip allows a lateral acceleration a_lat, a corner of curvature κ can be driven at most at

$$v^2|\kappa| \le a_{\text{lat}} \qquad\Longrightarrow\qquad v_{\max} = \sqrt{\frac{a_{\text{lat}}}{|\kappa|}} .$$

On μ = 1 the ceiling is a_lat = μg = 9.81 m/s², so `test-circuit`'s tightest corner (R ≈ 18 m) allows at most √(9.81 · 18) ≈ 13 m/s, and Unit 5's 20 m/s asked for 22 m/s² there: more than twice what the tyres have.

The agent needs the limit **before** the corner, so it uses Unit 2's instrument, the **curvature ahead**: the sharpest |κ| over the next 25 waypoints (about 50 m) from `current_index`, from Unit 2's Menger curvature.
`agents/unit-06.py` sets its speed goal to

$$v_{\text{goal}} = \min\!\Big(\texttt{TARGET\_SPEED},\ \sqrt{\texttt{A\_LAT} / |\kappa_{\text{ahead}}|}\Big)$$

(the straight's goal, `TARGET_SPEED` = 20 m/s, when κ_ahead is about 0), and feeds it to Unit 4's speed PI, which brakes when the goal is below the speed.
`A_LAT` is a knob: the lateral acceleration the cap allows, a little below what the tyres can do.

Here is the cap along `test-circuit`, computed from the waypoints exactly as the agent does, at `A_LAT` = 6 m/s²:

In [ ]:
def menger_curvature(xs, ys):
    # Unit 2: the signed curvature at each waypoint from (i - 1, i, i + 1), cyclic; left positive.
    n = len(xs)
    before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
    ax_, ay_, cx_, cy_ = xs[before], ys[before], xs[after], ys[after]
    twice_area = (xs - ax_) * (cy_ - ay_) - (ys - ay_) * (cx_ - ax_)
    sides = np.hypot(xs - ax_, ys - ay_) * np.hypot(cx_ - xs, cy_ - ys) * np.hypot(cx_ - ax_, cy_ - ay_)
    return 2.0 * twice_area / sides


def curvature_ahead(kappa, horizon=25):
    # Unit 2: at each waypoint i, the sharpest |kappa| over waypoints i .. i + horizon - 1 (cyclic).
    n = len(kappa)
    window = (np.arange(n)[:, None] + np.arange(horizon)[None, :]) % n
    return np.max(np.abs(kappa[window]), axis=1)


def speed_cap(kappa_ahead, a_lat, target=20.0):
    # The agent's goal at every waypoint: min(target, sqrt(a_lat / kappa_ahead)), the target where kappa_ahead is about 0.
    return np.array([min(target, math.sqrt(a_lat / k)) if k > 1e-9 else target for k in kappa_ahead])


track = load_track("test-circuit")
kappa = menger_curvature(track.x, track.y)
ahead = curvature_ahead(kappa)
cap = speed_cap(ahead, 6.0)
print(f"test-circuit: {len(kappa)} waypoints, {track.length:.0f} m; tightest corner R = {1.0 / np.max(np.abs(kappa)):.1f} m")
print(f"the cap at A_LAT = 6: lowest {np.min(cap):.2f} m/s; below 20 m/s on {np.mean(cap < 20.0) * 100:.0f}% of the waypoints")
print(f"the tightest corner at 20 m/s would need {20.0 ** 2 * np.max(np.abs(kappa)):.1f} m/s^2")
assert abs(np.min(cap) - math.sqrt(6.0 / np.max(np.abs(kappa)))) < 1e-9

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(track.s, cap, label="the cap, A_LAT = 6 m/s^2")
ax.plot(track.s, speed_cap(ahead, 9.81), label="the cap at mu g = 9.81 m/s^2")
ax.set_xlabel("distance along the lap (m)")
ax.set_ylabel("speed goal (m/s)")
ax.set_title("test-circuit: the curvature speed cap from 25 waypoints ahead")
ax.legend()
plt.show()

The cap starts to fall about 50 m before each corner (the horizon), stays down through it, and lifts as the corner leaves the window.

Now race it.
`agents/unit-06.py` ships `SPEED_MODE = "cap"` with `A_LAT = 6.0`; on `DynamicCar()`:

In [ ]:
capped = run_race(UNIT_06, "test-circuit", car=DynamicCar(), seed=0, max_seconds=1500.0)
print(f"the cap, A_LAT = 6, DynamicCar(): {summary(capped)}, {spins(capped)} spin-outs")
assert capped.finished and capped.collisions == 0 and spins(capped) == 0
assert abs(capped.total_time - 423.70) <= TICK

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, result in (("Unit 5 (flat 20 m/s)", spin_race), ("Unit 6 (the cap, A_LAT = 6)", capped)):
    early = result.telemetry["t"] < 150.0
    ax.plot(result.telemetry["t"][early], result.telemetry["v"][early], label=name)
ax.set_xlabel("t (s)")
ax.set_ylabel("v (m/s)")
ax.set_title("test-circuit on DynamicCar(): the first 150 s")
ax.legend()
plt.show()

Clean: three laps, no spin, no wall.
Unit 5's speed trace drops to zero at every spin-out (the car is held and respawned); Unit 6's dips **before** each corner and comes back after it.

## 4. How close to μg?

`A_LAT` = 6 m/s² is far below the 9.81 m/s² the tyres can give.
How far can it go?
The plan swept it on `DynamicCar()`, one deterministic race per cell, `max_seconds=1500`.
This is the **probe record (plan roar-006)**; the routed racekit test re-races every cell, and Unit 6's milestone re-races the two cells either side of the `test-circuit` edge:

| A_LAT (m/s²) | 5 | 6 | 7 | 8 | 9 | 9.25 | 9.5 | 9.75 | 10 |
|---|---|---|---|---|---|---|---|---|---|
| `test-circuit` | 433.95 | 423.70 | 417.65 | 413.55 | 411.05 | 410.50 | 410.05 | 36 spins | 37 spins |
| `monza-like` | 891.85 | 881.80 | 875.70 | 871.55 | 868.75 | 868.15 | 11 spins | 21 spins | 21 spins |

Every number is a clean race time in seconds, unless it says spins (then the spin count; that race did not finish).
The `oval` is 216.50 s at every A_LAT: there the cap never binds below 20 m/s.

Here is the last clean cell on `test-circuit`:

In [ ]:
edge = run_race(variant(UNIT_06, "unit-06-alat-9.5", A_LAT=9.5), "test-circuit", car=DynamicCar(), seed=0, max_seconds=1500.0)
print(f"A_LAT = 9.5, DynamicCar(): {summary(edge)}")
assert edge.finished and edge.collisions == 0 and spins(edge) == 0
assert abs(edge.total_time - 410.05) <= TICK
print(f"from A_LAT 6 to 9.5: {capped.total_time - edge.total_time:.2f} s faster over three laps; "
      f"max |offset| {max_offset(capped):.2f} m -> {max_offset(edge):.2f} m")
assert edge.total_time < capped.total_time and max_offset(edge) > max_offset(capped)

**The clean edge sits just under μg = 9.81 m/s²**: 9.5 on `test-circuit` (9.75 spins) and 9.25 on `monza-like` (9.5 spins).
That is the friction circle, seen through a whole race.
It is not exactly μg because the cap is an approximation of what the car does:

- the cap uses the **waypoints'** curvature, but the car drives pure pursuit's line, which is not the centreline (Unit 5: it cuts in at the entry and straightens early on the exit);
- the speed PI does not hit the goal exactly, and it brakes only once the corner is inside the 50 m window;
- every bit of throttle or brake in the corner uses part of the circle.

Notice also what the speed costs: from 6 to 9.5 the time falls by about 14 s, while the car moves further off the line.

## 5. The carla-like car: less grip than the model

`DynamicCar()` has μ = 1 everywhere.
The book's stand-in for the competition's simulator is `carla_like()`: a `DynamicCar` with its own parameters, which the book does **not** publish (Unit 8 identifies them from data), and a grip that varies along the track.
`carla_like(noise=False)` is deterministic; `carla_like(noise=True)` adds a seeded throttle jitter, like a real car's.

The same sweep on `carla_like(noise=False)` (**probe record (plan roar-006)**; the routed test re-races every cell):

| A_LAT (m/s²) | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|
| `test-circuit` | 434.70 | 424.30 | 418.40 | 19 spins | |
| `monza-like` | 892.60 | 882.45 | 876.30 | 872.15 | 21 spins |

The edge moves **down**, to 7 on `test-circuit` and 8 on `monza-like`.
The car has less grip than μ = 1 somewhere on each lap, and nothing in the cap knows where.
The flat Unit 5 agent spins 39 times on `test-circuit` on this car (probe record (plan roar-006); the milestone re-races it).

Here is `test-circuit`'s last clean cell:

In [ ]:
carla_edge = run_race(variant(UNIT_06, "unit-06-alat-7", A_LAT=7.0), "test-circuit", car=carla_like(noise=False),
                      seed=0, max_seconds=1500.0)
print(f"A_LAT = 7, carla_like(noise=False): {summary(carla_edge)}")
assert carla_edge.finished and carla_edge.collisions == 0 and spins(carla_edge) == 0
assert abs(carla_edge.total_time - 418.40) <= TICK

### The clean rule

A race that finishes with no collision but drives 3 m off the centreline is not a race you want to bet on: on a track 11–12 m wide that car is near the wall, and the real track's walls are not racekit's.
So from this unit on, a race is **clean** when

- it **finishes** with **0 collisions** (a spin-out counts as one), and
- its **max |cross-track error|** over the driving rows is **at most 2 m**.

Every clean race in this unit's sweeps stays under 1.2 m.

In [ ]:
def clean(result):
    # The book's clean rule from Unit 6 on.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET


for name, result in (("DynamicCar(), A_LAT 6", capped), ("DynamicCar(), A_LAT 9.5", edge),
                     ("carla_like(noise=False), A_LAT 7", carla_edge), ("DynamicCar(), flat (Unit 5)", spin_race)):
    print(f"{name:>34}: clean {clean(result)}, max |offset| {max_offset(result):.2f} m")
assert clean(capped) and clean(edge) and clean(carla_edge) and not clean(spin_race)

### The margin rule

The edge on `carla_like` is 7 on `test-circuit`.
Should the agent ship `A_LAT = 7`?
No, for two reasons.

- One clean race (or five, on five seeds) does not prove the next one; a noisy car or a different seed can tip a race at the edge over it.
- The carla-like car is still a model; the real simulator's grip is not racekit's.

So the book ships one grid step **inside** the clean edge: this is the **margin rule**.
`agents/unit-06.py` ships `A_LAT = 6.0`.
The reference on the noisy carla-like car, seed 0:

In [ ]:
reference = run_race(UNIT_06, "test-circuit", car=carla_like(noise=True), seed=0, max_seconds=1500.0)
print(f"agents/unit-06.py, carla_like(noise=True), seed 0: {summary(reference)}, clean {clean(reference)}")
assert clean(reference) and spins(reference) == 0
assert abs(reference.total_time - 422.55) <= TICK  # seed 0 of the probe record

Over seeds 0–4 on `carla_like(noise=True)`, `max_seconds=1500` (**probe record (plan roar-006)**; the routed test re-races it, and the milestone re-races seed 0 on every track):

| track | median (s) | spread (s) | clean |
|---|---|---|---|
| `oval` | 216.50 | 0.10 | 5/5 |
| `test-circuit` | 422.65 | 0.30 | 5/5 |
| `monza-like` | 880.50 | 0.15 | 5/5 |

All fifteen races clean, no spin-outs.
Exercise 6 asks what the margin buys, by racing the edge on the noisy car.

## 6. The bridge to Unit 7

The cap made the car safe.
It did not make it fast, and the reasons are built into it.

- **It never goes above 20 m/s.**
  On every straight the goal is `TARGET_SPEED`; raise it and the cap has to start braking much earlier than 50 m before a corner.
- **It brakes only when the corner is already close.**
  The horizon is 25 waypoints, about 50 m.
  How far does the car need to brake from a speed v to the cap of `test-circuit`'s tightest corner?

In [ ]:
A_BRAKE = 8.0  # m/s^2, Unit 3's full brake (a model number)
corner_cap = math.sqrt(6.0 / np.max(np.abs(kappa)))
horizon_m = float(np.median(np.hypot(np.diff(track.x), np.diff(track.y)))) * 25
for speed in (20.0, 30.0, 45.0):
    distance = (speed ** 2 - corner_cap ** 2) / (2.0 * A_BRAKE)
    print(f"from {speed:4.0f} m/s to {corner_cap:.2f} m/s at full brake: {distance:6.1f} m "
          f"({'inside' if distance < horizon_m else 'beyond'} the {horizon_m:.0f} m horizon)")
assert (20.0 ** 2 - corner_cap ** 2) / (2.0 * A_BRAKE) < horizon_m < (45.0 ** 2 - corner_cap ** 2) / (2.0 * A_BRAKE)

At 20 m/s the car can stop for the tightest corner inside the window; at 45 m/s it needs more than twice the window, and a full brake uses most of the friction circle that the corner entry also needs.
A faster car has to **plan** its speed: work out, for every point of the lap, the fastest speed from which it can still slow down in time for every corner ahead.

That is Unit 7: a speed **profile** along the whole lap, a braking pass backwards from each corner, and a search for the grip that is really there, section by section.

## Summary

- The **dynamic bicycle** makes lateral force from **slip angles**; at low speed it agrees with Unit 3's kinematic bicycle.
- Each tyre's force **saturates**: the **friction circle** a_lat² + a_long² ≤ (μg)² is a hard ceiling, and past it the car spins.
- The **cornering speed limit** v_max = √(a_lat / |κ|), with Unit 2's curvature ahead, turns the ceiling into a speed goal.
- On `DynamicCar()` the clean edge sits just under μg; on `carla_like` it is lower, so the real car's grip is lower than the model's.
- A race is **clean** when it finishes with 0 collisions and stays within 2 m of the line; the shipped setting is one grid step inside the clean edge (**the margin rule**).
- `agents/unit-06.py` ships the cap at `A_LAT = 6.0`: clean on every track, and slow.
  Unit 7 makes it fast.